# TP 1: LDA/QDA y optimización matemática de modelos

Se puede consultar la introducción teórica en el mono-notebook, se prefiere mantener este lo más chico posible.

In [34]:
# imports
import numpy as np
import numpy.linalg as LA

from base.qda import QDA, TensorizedQDA
from base.cholesky import QDA_Chol1, QDA_Chol2, QDA_Chol3
from utils.bench import Benchmark
from utils.datasets import (get_iris_dataset, get_letters_dataset, 
                            get_penguins_dataset, get_wine_dataset,
                            label_encode)


## Ejemplo

In [35]:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()

X_full.shape, y_full.shape

((178, 13), (178, 1))

In [36]:
# encodeamos a número las clases
y_full_encoded = label_encode(y_full)

y_full[:5], y_full_encoded[:5]

(array([['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0']], dtype='<U7'),
 array([[0],
        [0],
        [0],
        [0],
        [0]]))

In [37]:
# generamos el benchmark
# observar que son valores muy bajos de runs para que corra rápido ahora
b = Benchmark(
    X_full, y_full_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


In [38]:
# bencheamos un par
to_bench = [QDA]

for model in to_bench:
    b.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [39]:
# como es una clase, podemos seguir bencheando más después
b.bench(TensorizedQDA)

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [40]:
# hacemos un summary
b.summary()

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb
model,,,,,,,,,
QDA,0.68365,0.299558,5.5748,0.756486,0.982407,0.018913,0.063395,0.008547,0.064735
TensorizedQDA,0.69885,0.300021,2.3284,0.374260,0.982593,0.018570,0.049281,0.012646,0.050401


In [41]:
# son muchos datos! nos quedamos con un par nomás
summ = b.summary()

# como es un pandas DataFrame, subseteamos columnas fácil
summ[['train_median_ms', 'test_median_ms','mean_accuracy']]

,train_median_ms,test_median_ms,mean_accuracy
model,,,
QDA,0.68365,5.5748,0.982407
TensorizedQDA,0.69885,2.3284,0.982593


In [42]:
# podemos setear un baseline para que fabrique columnas de comparación
summ = b.summary(baseline='QDA')

summ

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,0.68365,0.299558,5.5748,0.756486,0.982407,0.018913,0.063395,0.008547,0.064735,1.00000,1.000000,1.000000,1.000000
TensorizedQDA,0.69885,0.300021,2.3284,0.374260,0.982593,0.018570,0.049281,0.012646,0.050401,0.97825,2.394262,1.018488,0.675867


In [43]:
# volvemos a subsetear columnas
summ[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.68365,5.5748,0.982407,1.00000,1.000000,1.000000,1.000000
TensorizedQDA,0.69885,2.3284,0.982593,0.97825,2.394262,1.018488,0.675867


## Tensorización

### 1) Diferencias entre `QDA` y `TensorizedQDA`

### 1.1 ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?

`TensorizedQDA` paraleliza las operaciones sobre las $k$ clases mediante tensorización, pero continúa procesando individualmente las $n$ observaciones a predecir.

Durante el entrenamiento, QDA calcula para cada clase su vector de medias $\mu_k$ y su matriz de covarianza $\Sigma_k$. Durante la predicción, cada nueva observación $x$ debe ser evaluada respecto de cada clase utilizando, entre otros términos, la forma cuadrática:

$$
(x-\mu_k)^T\Sigma_k^{-1}(x-\mu_k)
$$

En `QDA`, para una misma observación, este cálculo se realiza individualmente para cada clase. En `TensorizedQDA`, en cambio, las medias y las matrices de covarianza inversa correspondientes a las distintas clases se apilan en tensores, permitiendo realizar estas operaciones para todas las clases mediante operaciones matriciales de NumPy.

Por ejemplo, si se tienen $n=4$ observaciones y $k=3$ clases, conceptualmente `QDA` realiza:

    x1 → C1 → C2 → C3
    x2 → C1 → C2 → C3
    x3 → C1 → C2 → C3
    x4 → C1 → C2 → C3

mientras que `TensorizedQDA` realiza:

    x1 → [C1, C2, C3]
    x2 → [C1, C2, C3]
    x3 → [C1, C2, C3]
    x4 → [C1, C2, C3]

Por lo tanto, `TensorizedQDA` elimina el ciclo explícito sobre las $k$ clases, pero mantiene el ciclo sobre las $n$ observaciones. Es decir, la tensorización implementada en esta clase actúa sobre la dimensión correspondiente a las clases, no sobre la dimensión correspondiente a las observaciones.

### 1.2 Análisis de los `shapes` de `tensor_inv_cov` y `tensor_means`

> Nota: en la consigna se menciona `tensor_inv_covs`, mientras que en la implementación el atributo se denomina `tensor_inv_cov`.

En `QDA`, para cada una de las $k$ clases se calcula un vector de medias y una matriz de covarianza inversa.

Para una clase $j$:

$$
\mu_j \in \mathbb{R}^{p \times 1}
$$

$$
\Sigma_j^{-1} \in \mathbb{R}^{p \times p}
$$

donde $p$ es la cantidad de features.

En `TensorizedQDA`, luego de ejecutar el `_fit_params` de `QDA`, estos elementos se apilan mediante:

    self.tensor_inv_cov = np.stack(self.inv_covs)
    self.tensor_means = np.stack(self.means)

Como existe una media y una matriz de covarianza inversa para cada una de las $k$ clases, los shapes resultantes son:

$$
\text{tensor\_inv\_cov.shape}=(k,p,p)
$$

$$
\text{tensor\_means.shape}=(k,p,1)
$$

Es decir, la nueva primera dimensión corresponde a las $k$ clases.

Durante la predicción, el método `predict` toma cada observación de forma individual y la transforma en un vector columna mediante:

    X[:, i].reshape(-1, 1)

por lo que, para una observación $x$:

$$
x.shape=(p,1)
$$

A continuación, `TensorizedQDA` ejecuta:

    unbiased_x = x - self.tensor_means

Como `x` tiene shape $(p,1)$ y `tensor_means` tiene shape $(k,p,1)$, NumPy utiliza broadcasting para restar la misma observación a la media de cada una de las $k$ clases. El resultado tiene:

$$
\text{unbiased\_x.shape}=(k,p,1)
$$

y contiene conceptualmente:

$$
\begin{bmatrix}
x-\mu_1 \\
x-\mu_2 \\
\vdots \\
x-\mu_k
\end{bmatrix}
$$

Luego se ejecuta:

    unbiased_x.transpose(0,2,1)

El `transpose` mantiene la dimensión correspondiente a las clases e intercambia las dos últimas dimensiones:

$$
(k,p,1)\rightarrow(k,1,p)
$$

Esto permite realizar de forma tensorizada la misma forma cuadrática que `QDA` calcula individualmente para cada clase:

$$
(x-\mu_j)^T\Sigma_j^{-1}(x-\mu_j)
$$

En el código:

    inner_prod = unbiased_x.transpose(0,2,1) @ self.tensor_inv_cov @ unbiased_x

Los shapes involucrados son:

$$
(k,1,p)@(k,p,p)@(k,p,1)
$$

La primera multiplicación produce:

$$
(k,1,p)@(k,p,p)\rightarrow(k,1,p)
$$

y la segunda:

$$
(k,1,p)@(k,p,1)\rightarrow(k,1,1)
$$

Por lo tanto:

$$
\text{inner\_prod.shape}=(k,1,1)
$$

Cada una de las $k$ matrices de dimensión $(1,1)$ contiene la forma cuadrática correspondiente a una clase:

$$
(x-\mu_j)^T\Sigma_j^{-1}(x-\mu_j)
$$

Luego el código aplica:

    inner_prod.flatten()

por lo que:

$$
(k,1,1)\rightarrow(k,)
$$

obteniéndose un vector con un valor para cada clase.

Por otro lado:

    LA.det(self.tensor_inv_cov)

calcula el determinante de cada una de las $k$ matrices de covarianza inversa. Como `tensor_inv_cov` tiene shape $(k,p,p)$, el resultado tiene shape:

$$
(k,)
$$

Así, la expresión:

    0.5*np.log(LA.det(self.tensor_inv_cov)) - 0.5*inner_prod.flatten()

devuelve un vector de shape $(k,)$ que contiene un score log-condicional para cada una de las $k$ clases.

Finalmente, en `_predict_one` se ejecuta:

    np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

`log_a_priori` también tiene un valor por clase, por lo que su shape es $(k,)$. Al sumarlo a los scores log-condicionales se obtiene un score a posteriori para cada clase, y `np.argmax` devuelve el índice de la clase con mayor valor.

En resumen, la evolución de los shapes para una observación es:

| Elemento | Shape |
|---|---|
| `x` | $(p,1)$ |
| `self.means[j]` | $(p,1)$ |
| `self.inv_covs[j]` | $(p,p)$ |
| `tensor_means` | $(k,p,1)$ |
| `tensor_inv_cov` | $(k,p,p)$ |
| `unbiased_x` | $(k,p,1)$ |
| `unbiased_x.transpose(0,2,1)` | $(k,1,p)$ |
| primera multiplicación matricial | $(k,1,p)$ |
| `inner_prod` | $(k,1,1)$ |
| `inner_prod.flatten()` | $(k,)$ |
| `LA.det(self.tensor_inv_cov)` | $(k,)$ |
| scores log-condicionales | $(k,)$ |
| scores a posteriori | $(k,)$ |
| `np.argmax(...)` | escalar |

Por lo tanto, `TensorizedQDA` llega a la misma predicción que `QDA` porque no modifica el cálculo matemático realizado para cada clase. `QDA` calcula cada forma cuadrática por separado recorriendo las clases, mientras que `TensorizedQDA` apila los parámetros de las $k$ clases y realiza esos mismos cálculos conjuntamente mediante operaciones tensorizadas. En consecuencia, se obtienen los mismos scores por clase y el `argmax` selecciona la misma clase.

### 2) Optimización

#### 3. Implementación de `FasterQDA`

Se implementa `FasterQDA` heredando de `TensorizedQDA`, pero redefiniendo el método `predict` para procesar simultáneamente las $n$ observaciones y eliminar el ciclo `for` presente en `BaseBayesianClassifier.predict`.

In [44]:
class FasterQDA(TensorizedQDA):

    def predict(self, X):

        # X: (p, n)
        # tensor_means: (k, p, 1)
        # broadcasting -> (k, p, n)
        unbiased_X = X - self.tensor_means

        # (k, n, p) @ (k, p, p) @ (k, p, n)
        # -> (k, n, n)
        inner_prod = (
            unbiased_X.transpose(0, 2, 1)
            @ self.tensor_inv_cov
            @ unbiased_X
        )

        # Para cada clase sólo interesan los términos
        # correspondientes a cada observación consigo misma.
        # (k, n, n) -> (k, n)
        quad_terms = np.diagonal(
            inner_prod,
            axis1=1,
            axis2=2
        )

        # determinantes: (k,) -> (k, 1)
        log_conditionals = (
            0.5 * np.log(LA.det(self.tensor_inv_cov))[:, None]
            - 0.5 * quad_terms
        )

        # log_a_priori: (k,) -> (k, 1)
        # resultado: (k, n)
        log_posteriori = (
            self.log_a_priori[:, None]
            + log_conditionals
        )

        # elegimos la clase de mayor score para cada observación
        # (k, n) -> (n,) -> (1, n)
        return np.argmax(log_posteriori, axis=0).reshape(1, -1)

In [45]:
qda = QDA()
tensorized_qda = TensorizedQDA()
faster_qda = FasterQDA()

qda.fit(X_full.T, y_full_encoded.T)
tensorized_qda.fit(X_full.T, y_full_encoded.T)
faster_qda.fit(X_full.T, y_full_encoded.T)

pred_qda = qda.predict(X_full.T)
pred_tensorized = tensorized_qda.predict(X_full.T)
pred_faster = faster_qda.predict(X_full.T)

print("QDA == TensorizedQDA:",
      np.array_equal(pred_qda, pred_tensorized))

print("QDA == FasterQDA:",
      np.array_equal(pred_qda, pred_faster))

QDA == TensorizedQDA: True
QDA == FasterQDA: True
